1. Load Data
2. Exploratory Data Analysis (EDA)
   - Univariate
   - Bivariate
3. Preprocessing
4. Feature Engineering
5. Train-Test Split
6. Model Training (multiple models)
7. Evaluation
8. Optimization

In [1]:
import pandas as pd

df = pd.read_csv("fall_detection_dataset.csv")

print(df.head())
print(df.info())

      acc_x     acc_y     acc_z    gyro_x    gyro_y    gyro_z  heart_rate  \
0  0.496714  1.399355  9.124822 -9.539038 -4.317468 -2.118798          97   
1 -0.138264  0.924634  9.655481 -4.301925 -0.156017 -2.267071         100   
2  0.647689  0.059630  9.007580 -2.068028  0.090084 -8.978216          66   
3  1.523030 -0.646937  9.492038  9.438438  2.363152 -1.650451          73   
4 -0.234153  0.698223  7.906385  2.782766 -6.834292  3.664145         102   

   room_temp  room_light   room_type     posture time_of_day risk_level  \
0         23         689     Hallway       Lying     Morning     Medium   
1         18         177    Bathroom  Transition     Morning        Low   
2         22         271  LivingRoom    Standing     Evening     Medium   
3         27         388     Hallway       Lying   Afternoon     Medium   
4         23         644     Hallway     Walking       Night     Medium   

   fall_event fall_severity  
0           0           NaN  
1           1          Mil

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   acc_x          1000 non-null   float64
 1   acc_y          1000 non-null   float64
 2   acc_z          1000 non-null   float64
 3   gyro_x         1000 non-null   float64
 4   gyro_y         1000 non-null   float64
 5   gyro_z         1000 non-null   float64
 6   heart_rate     1000 non-null   int64  
 7   room_temp      1000 non-null   int64  
 8   room_light     1000 non-null   int64  
 9   room_type      1000 non-null   object 
 10  posture        1000 non-null   object 
 11  time_of_day    1000 non-null   object 
 12  risk_level     1000 non-null   object 
 13  fall_event     1000 non-null   int64  
 14  fall_severity  87 non-null     object 
dtypes: float64(6), int64(4), object(5)
memory usage: 117.3+ KB


In [3]:
df.isnull().sum()

acc_x              0
acc_y              0
acc_z              0
gyro_x             0
gyro_y             0
gyro_z             0
heart_rate         0
room_temp          0
room_light         0
room_type          0
posture            0
time_of_day        0
risk_level         0
fall_event         0
fall_severity    913
dtype: int64

In [4]:
df["fall_severity"].unique()

array([nan, 'Mild', 'Severe', 'Moderate'], dtype=object)

In [5]:
df['fall_severity'] = df['fall_severity'].fillna('No Fall')

In [6]:
df.isnull().sum()

acc_x            0
acc_y            0
acc_z            0
gyro_x           0
gyro_y           0
gyro_z           0
heart_rate       0
room_temp        0
room_light       0
room_type        0
posture          0
time_of_day      0
risk_level       0
fall_event       0
fall_severity    0
dtype: int64

In [7]:
# Check data types of each column
print(df.dtypes)

acc_x            float64
acc_y            float64
acc_z            float64
gyro_x           float64
gyro_y           float64
gyro_z           float64
heart_rate         int64
room_temp          int64
room_light         int64
room_type         object
posture           object
time_of_day       object
risk_level        object
fall_event         int64
fall_severity     object
dtype: object


In [8]:
df["room_type"].unique()

array(['Hallway', 'Bathroom', 'LivingRoom', 'Bedroom'], dtype=object)

In [9]:
df["posture"].unique()

array(['Lying', 'Transition', 'Standing', 'Walking', 'Sitting'],
      dtype=object)

In [10]:
df["time_of_day"].unique()

array(['Morning', 'Evening', 'Afternoon', 'Night'], dtype=object)

In [11]:
df["risk_level"].unique()

array(['Medium', 'Low', 'High'], dtype=object)

In [12]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import pandas as pd

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# OneHot columns (safe check)
onehot_cols = [col for col in ['room_type', 'posture', 'time_of_day'] if col in df.columns]

ohe = OneHotEncoder(drop='first', sparse_output=False)

encoded_data = ohe.fit_transform(df[onehot_cols])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=ohe.get_feature_names_out(onehot_cols),
    index=df.index
)

df = df.drop(columns=onehot_cols)
df = pd.concat([df, encoded_df], axis=1)

# Ordinal Encoding
ordinal_col = ['risk_level']

ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])

df['risk_level_encoded'] = ordinal_encoder.fit_transform(df[ordinal_col])

df = df.drop(columns=ordinal_col)

print(df.head())

      acc_x     acc_y     acc_z    gyro_x    gyro_y    gyro_z  heart_rate  \
0  0.496714  1.399355  9.124822 -9.539038 -4.317468 -2.118798          97   
1 -0.138264  0.924634  9.655481 -4.301925 -0.156017 -2.267071         100   
2  0.647689  0.059630  9.007580 -2.068028  0.090084 -8.978216          66   
3  1.523030 -0.646937  9.492038  9.438438  2.363152 -1.650451          73   
4 -0.234153  0.698223  7.906385  2.782766 -6.834292  3.664145         102   

   room_temp  room_light  fall_event  ... room_type_Hallway  \
0         23         689           0  ...               1.0   
1         18         177           1  ...               0.0   
2         22         271           0  ...               0.0   
3         27         388           0  ...               1.0   
4         23         644           0  ...               1.0   

   room_type_LivingRoom  posture_Sitting  posture_Standing  \
0                   0.0              0.0               0.0   
1                   0.0           

In [13]:
df

,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,heart_rate,room_temp,room_light,fall_event,...,room_type_Hallway,room_type_LivingRoom,posture_Sitting,posture_Standing,posture_Transition,posture_Walking,time_of_day_Evening,time_of_day_Morning,time_of_day_Night,risk_level_encoded
0,0.496714,1.399355,9.124822,-9.539038,-4.317468,-2.118798,97,23,689,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,-0.138264,0.924634,9.655481,-4.301925,-0.156017,-2.267071,100,18,177,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.647689,0.059630,9.007580,-2.068028,0.090084,-8.978216,66,22,271,0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
3,1.523030,-0.646937,9.492038,9.438438,2.363152,-1.650451,73,27,388,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.234153,0.698223,7.906385,2.782766,-6.834292,3.664145,102,23,644,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.281100,1.070150,9.877481,0.142288,-0.244825,0.857347,75,19,978,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
996,1.797687,-0.026521,10.057753,-10.389059,3.557053,5.763241,101,29,203,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
997,0.640843,-0.881875,8.558239,-1.601489,15.564551,-6.087019,101,24,944,1,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
998,-0.571179,-0.163067,10.134176,8.216891,4.040181,2.339752,110,24,303,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [14]:
# fall_event and fall_severity are already in df
# If you want to explicitly keep them at the front of the dataframe:
cols = ['fall_event', 'fall_severity'] + [col for col in df.columns if col not in ['fall_event', 'fall_severity']]
df = df[cols]

print(df.head())

   fall_event fall_severity     acc_x     acc_y     acc_z    gyro_x    gyro_y  \
0           0       No Fall  0.496714  1.399355  9.124822 -9.539038 -4.317468   
1           1          Mild -0.138264  0.924634  9.655481 -4.301925 -0.156017   
2           0       No Fall  0.647689  0.059630  9.007580 -2.068028  0.090084   
3           0       No Fall  1.523030 -0.646937  9.492038  9.438438  2.363152   
4           0       No Fall -0.234153  0.698223  7.906385  2.782766 -6.834292   

     gyro_z  heart_rate  room_temp  ...  room_type_Hallway  \
0 -2.118798          97         23  ...                1.0   
1 -2.267071         100         18  ...                0.0   
2 -8.978216          66         22  ...                0.0   
3 -1.650451          73         27  ...                1.0   
4  3.664145         102         23  ...                1.0   

   room_type_LivingRoom  posture_Sitting  posture_Standing  \
0                   0.0              0.0               0.0   
1               

In [15]:
df

,fall_event,fall_severity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,heart_rate,room_temp,...,room_type_Hallway,room_type_LivingRoom,posture_Sitting,posture_Standing,posture_Transition,posture_Walking,time_of_day_Evening,time_of_day_Morning,time_of_day_Night,risk_level_encoded
0,0,No Fall,0.496714,1.399355,9.124822,-9.539038,-4.317468,-2.118798,97,23,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,1,Mild,-0.138264,0.924634,9.655481,-4.301925,-0.156017,-2.267071,100,18,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0,No Fall,0.647689,0.059630,9.007580,-2.068028,0.090084,-8.978216,66,22,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0,No Fall,1.523030,-0.646937,9.492038,9.438438,2.363152,-1.650451,73,27,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0,No Fall,-0.234153,0.698223,7.906385,2.782766,-6.834292,3.664145,102,23,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,No Fall,-0.281100,1.070150,9.877481,0.142288,-0.244825,0.857347,75,19,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
996,0,No Fall,1.797687,-0.026521,10.057753,-10.389059,3.557053,5.763241,101,29,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
997,1,Mild,0.640843,-0.881875,8.558239,-1.601489,15.564551,-6.087019,101,24,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
998,1,Moderate,-0.571179,-0.163067,10.134176,8.216891,4.040181,2.339752,110,24,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [16]:
df["fall_event"].value_counts()

fall_event
0    913
1     87
Name: count, dtype: int64

In [19]:
from sklearn.preprocessing import OrdinalEncoder

severity_order = [['No Fall', 'Mild', 'Moderate', 'Severe']]

severity_encoder = OrdinalEncoder(categories=severity_order)

if 'fall_severity' in df.columns:
	df['fall_severity_encoded'] = severity_encoder.fit_transform(df[['fall_severity']])
	df = df.drop(columns=['fall_severity'])
else:
	print("Column 'fall_severity' not found. Using existing 'fall_severity_encoded' if already created.")

Column 'fall_severity' not found. Using existing 'fall_severity_encoded' if already created.


In [20]:
df

,fall_event,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,heart_rate,room_temp,room_light,...,room_type_LivingRoom,posture_Sitting,posture_Standing,posture_Transition,posture_Walking,time_of_day_Evening,time_of_day_Morning,time_of_day_Night,risk_level_encoded,fall_severity_encoded
0,0,0.496714,1.399355,9.124822,-9.539038,-4.317468,-2.118798,97,23,689,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1,-0.138264,0.924634,9.655481,-4.301925,-0.156017,-2.267071,100,18,177,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0,0.647689,0.059630,9.007580,-2.068028,0.090084,-8.978216,66,22,271,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0,1.523030,-0.646937,9.492038,9.438438,2.363152,-1.650451,73,27,388,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0,-0.234153,0.698223,7.906385,2.782766,-6.834292,3.664145,102,23,644,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,-0.281100,1.070150,9.877481,0.142288,-0.244825,0.857347,75,19,978,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
996,0,1.797687,-0.026521,10.057753,-10.389059,3.557053,5.763241,101,29,203,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
997,1,0.640843,-0.881875,8.558239,-1.601489,15.564551,-6.087019,101,24,944,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
998,1,-0.571179,-0.163067,10.134176,8.216891,4.040181,2.339752,110,24,303,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Apply StandardScaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

# Check shapes
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (800, 20)
X_test_scaled: (200, 20)


In [24]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Define model
svc = SVC()

# Define parameter grid
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}

# GridSearchCV
grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring='recall',   # VERY IMPORTANT
    cv=5,
    n_jobs=-1
)

# Fit
grid.fit(X_train_scaled, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best model
best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test_scaled)

# Evaluation
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Confusion Matrix:
 [[178   5]
 [ 17   0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.97      0.94       183
           1       0.00      0.00      0.00        17

    accuracy                           0.89       200
   macro avg       0.46      0.49      0.47       200
weighted avg       0.84      0.89      0.86       200



In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Model
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)

# Parameter grid
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

# GridSearch
grid = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='recall',   # VERY IMPORTANT
    cv=5,
    n_jobs=-1
)

# Fit (no scaling needed)
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best model
best_dt = grid.best_estimator_

# Predictions
y_pred = best_dt.predict(X_test)

# Evaluation
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}

Confusion Matrix:
 [[89 94]
 [ 5 12]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.49      0.64       183
           1       0.11      0.71      0.20        17

    accuracy                           0.51       200
   macro avg       0.53      0.60      0.42       200
weighted avg       0.88      0.51      0.60       200



In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Model
rf = RandomForestClassifier(class_weight='balanced', random_state=42)

# Parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# GridSearch
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='recall',   # VERY IMPORTANT
    cv=5,
    n_jobs=-1
)

# Fit (NO scaling needed)
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best model
best_rf = grid.best_estimator_

# Predictions
y_pred = best_rf.predict(X_test)

# Evaluation
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

Confusion Matrix:
 [[180   3]
 [ 17   0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.98      0.95       183
           1       0.00      0.00      0.00        17

    accuracy                           0.90       200
   macro avg       0.46      0.49      0.47       200
weighted avg       0.84      0.90      0.87       200



In [30]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Fix 1: define ratio
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# Model
xgb = XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

# Smaller grid (safe)
param_grid = {
    'n_estimators': [100],
    'max_depth': [3, 5],
    'learning_rate': [0.1]
}

# GridSearch
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1
)

# Fit
grid.fit(X_train, y_train)

# Best model
best_xgb = grid.best_estimator_

# Predict
y_pred = best_xgb.predict(X_test)

# Evaluation
print("Best Parameters:", grid.best_params_)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Confusion Matrix:
 [[160  23]
 [ 16   1]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.87      0.89       183
           1       0.04      0.06      0.05        17

    accuracy                           0.81       200
   macro avg       0.48      0.47      0.47       200
weighted avg       0.84      0.81      0.82       200

